# Case Study – Data Modeling in [Neo4j](https://neo4j.com/)

## Setup

### Running Neo4j in Docker

- [Docker Hub: neo4j](https://hub.docker.com/_/neo4j)

In [ ]:
# !docker pull neo4j:2025.11.2-ubi9
# !docker run name neo4j -p 7474:7474 -p 7687:7687 -e NEO4J_AUTH=neo4j/test2025 neo4j:latest

### Install Dependencies

In [ ]:
%pip install neo4j

### Imports

In [ ]:
import pandas as pd
from neo4j import GraphDatabase

### Connect to Neo4j

In [ ]:
URI = "bolt://localhost:7687"
AUTH = ("neo4j", "test2025")

driver = GraphDatabase.driver(URI, auth=AUTH)

### Functions

In [ ]:
def run_query(cypher: str, params=None):
    params = params or {}
    with driver.session() as session:
        result = session.run(cypher, params)
        return pd.DataFrame([r.data() for r in result])

## Run Queries

List all users

In [ ]:
list_all_users_query = """
MATCH (u:User)
RETURN u.id, u.name, u.age, u.email;
"""

run_query(cypher=list_all_users_query)

List each user's friends

In [ ]:
list_user_friends_query = """
MATCH (u:User {id: $userId})-[:FRIENDS_WITH]-(friend:User)
RETURN friend.name;
"""

Alice (id: 1)

In [ ]:
run_query(cypher=list_user_friends_query, params={"userId": 1})

Bob (id: 2)

In [ ]:
run_query(cypher=list_user_friends_query, params={"userId": 2})

Carol (id: 3)

In [ ]:
run_query(cypher=list_user_friends_query, params={"userId": 3})

List all films along with their genre

In [ ]:
list_films_query = """
MATCH (f:Film)-[:IN_GENRE]->(g:Genre)
RETURN f.title AS film, g.name AS genre
ORDER BY film;
"""

run_query(cypher=list_films_query)

List which films each user watched (with scores)

In [ ]:
list_user_watched_films_query = """
MATCH (u:User)-[w:WATCHED]->(f:Film)
RETURN u.name AS user, f.title AS film, w.score AS score
ORDER BY user, score DESC;
"""

run_query(cypher=list_user_watched_films_query)

List films that friends liked but the respective user haven't watched yet

In [ ]:
list_unwatched_films_by_user_query = """
MATCH (me:User {id: $userId})-[:FRIENDS_WITH]-(friend:User)
MATCH (friend)-[w:WATCHED]->(f:Film)
WHERE w.score >= 4
  AND NOT (me)-[:WATCHED]->(f)
RETURN
  f.id    AS filmId,
  f.title AS title,
  f.year  AS year,
  avg(w.score)      AS avgFriendScore,
  count(DISTINCT friend) AS friendsWhoLiked
ORDER BY friendsWhoLiked DESC, avgFriendScore DESC, year DESC;
"""

run_query(cypher=list_unwatched_films_by_user_query, params={"userId": 1})  # Alice

List each user's watchlist

In [ ]:
list_users_watchlist_query = """
MATCH (u:User)-[r:WANTS_TO_WATCH]->(f:Film)
RETURN u.name AS user, f.title AS film
ORDER BY user, film ASC;
"""

run_query(cypher=list_users_watchlist_query)

Recommend films based on friend's watchlist

In [ ]:
recommend_films_query = """
MATCH (me:User {id: $userId})-[:FRIENDS_WITH]-(friend:User)
MATCH (friend)-[:WANTS_TO_WATCH]->(f:Film)
WHERE NOT (me)-[:WATCHED]->(f)
  AND NOT (me)-[:WANTS_TO_WATCH]->(f)
RETURN f.title AS recommended, count(DISTINCT friend) AS friendsInterested
ORDER BY friendsInterested DESC, recommended;
"""

run_query(cypher=recommend_films_query, params={"userId": 3})  # Carol

## ASCII Schema

---

[pt-BR]

## Responder

Q. **Explicar por que Neo4j é superior para consultas altamente relacionais.**

R. O banco em questão é considerado *superior* por ser mais simples, rápido e previsível quando o assunto é relações profundas.

Por exemplo: Em um banco de dados *tradicional* (SQL), consultas altamente relacionais acabam, frequentemente, gerando muitos JOINs e subqueries. Isso se dá pelo fato dos dados estarem estruturados em tabelas, onde as relações são feitas via chaves-estrangeiras (FKs).

Diferente disso, o Neo4j é um banco de dados orientado a grafos, onde, cada informação é armazenada em nós. E uma vez que cada relação é estabelecida, ir para os nós vizinhos é muito menos custoso. A causa disso é porque cada nó guarda os respectivos ponteiros para cada uma das suas relações, e cada relação aponta para seus respectivos nós. Ou seja, não é preciso "refazer" as relações através de JOINs, basta apenas seguir os ponteiros.